# 03 — Create Gold Tables

Load silver stock price data and build analytics-ready gold tables.

**Gold outputs:**
- `gold_stock_prices.csv` — one row per ticker with returns, volatility, and volume averages
- `gold_moving_averages.csv` — daily prices with 20-day and 50-day moving averages
- `gold_volume_summary.csv` — one row per ticker with volume stats

**Flow:** silver CSV → gold metrics → inspect → save gold CSVs

**Prerequisite:** Run `02_clean_silver.ipynb` first so `data/processed/silver/silver_stock_prices.csv` exists.

## Setup

Add `src` to the Python path and import the gold metrics helpers.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
SILVER_INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "silver" / "silver_stock_prices.csv"
GOLD_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "gold"

PERFORMANCE_OUTPUT_PATH = GOLD_OUTPUT_DIR / "gold_stock_prices.csv"
MOVING_AVERAGES_OUTPUT_PATH = GOLD_OUTPUT_DIR / "gold_moving_averages.csv"
VOLUME_SUMMARY_OUTPUT_PATH = GOLD_OUTPUT_DIR / "gold_volume_summary.csv"

sys.path.insert(0, str(SRC_DIR))

from metrics import (
    create_gold_moving_averages,
    create_gold_performance_summary,
    create_gold_volume_summary,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Silver input: {SILVER_INPUT_PATH}")
print(f"Gold output dir: {GOLD_OUTPUT_DIR}")

## Load silver data

Read the cleaned silver file created by the silver notebook.

In [ ]:
if not SILVER_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Silver file not found at {SILVER_INPUT_PATH}. Run 02_clean_silver.ipynb first."
    )

silver_df = pd.read_csv(SILVER_INPUT_PATH, parse_dates=["date", "loaded_at"])

print(f"Silver rows: {len(silver_df):,}")
print(f"Tickers: {sorted(silver_df['ticker'].unique())}")
silver_df.head()

## Gold stock performance summary

One row per ticker with start/end prices, total return, average daily return, volatility, and average volume metrics.

In [ ]:
gold_performance_df = create_gold_performance_summary(silver_df)

print(f"Performance rows: {len(gold_performance_df):,}")
gold_performance_df

## Gold moving averages

Daily prices with 20-day and 50-day moving averages per ticker.

In [ ]:
gold_moving_averages_df = create_gold_moving_averages(silver_df)

print(f"Moving average rows: {len(gold_moving_averages_df):,}")
gold_moving_averages_df.head(10)

## Gold volume summary

One row per ticker with min, max, total, and average volume metrics.

In [ ]:
gold_volume_summary_df = create_gold_volume_summary(silver_df)

print(f"Volume summary rows: {len(gold_volume_summary_df):,}")
gold_volume_summary_df

## Save gold tables

Write all gold outputs to `data/processed/gold/`.

In [ ]:
GOLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gold_performance_df.to_csv(PERFORMANCE_OUTPUT_PATH, index=False)
gold_moving_averages_df.to_csv(MOVING_AVERAGES_OUTPUT_PATH, index=False)
gold_volume_summary_df.to_csv(VOLUME_SUMMARY_OUTPUT_PATH, index=False)

print(f"Saved performance summary to {PERFORMANCE_OUTPUT_PATH}")
print(f"Saved moving averages to {MOVING_AVERAGES_OUTPUT_PATH}")
print(f"Saved volume summary to {VOLUME_SUMMARY_OUTPUT_PATH}")